In [1]:
import re
import jieba
import jieba.posseg as pseg
from pygments.lexer import words
from sudachipy.sudachipy import SplitMode
from yaml import tokens

try:
    from sudachipy import tokenizer as sudachi_tokenizer
    from sudachipy import dictionary
    SUDACHI_AVAILABLE = True
    sudachi_dict = dictionary.Dictionary().create()
    sudachi_mode = sudachi_tokenizer.Tokenizer.SplitMode.C
except ImportError:
    SUDACHI_AVAILABLE = False
    print("Warning: Sudachi not available. Install with: pip install sudachipy sudachidict-core")

try:
    from konlpy.tag import Okt
    # Test if Java is actually available
    try:
        # source ~/.zshrc or ~/.bashrc to ensure JAVA_HOME is set, if needed
        okt = Okt()
        okt.morphs("테스트")
        KONLPY_AVAILABLE = True
    except:
        KONLPY_AVAILABLE = False
        print("Warning: KoNLPy installed but Java not available. Using fallback for Korean.")
except ImportError:
    KONLPY_AVAILABLE = False

try:
    from pythainlp.tokenize import word_tokenize as thai_tokenize
    PYTHAINLP_AVAILABLE = True
except ImportError:
    PYTHAINLP_AVAILABLE = False

In [2]:
import os
import json
from google import genai
from google.genai import types
from dotenv import load_dotenv
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [3]:
# load ecklectic data
import pandas as pd
df = pd.read_csv("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/raw/eclektic_7.csv")

# generate a new column 'eclektic_id' by concatenating 'language' string,'q_id' int64
df[f"eclektic_id"] = df["language"] + "_" + df["q_id"].astype(str)

df.head()

,original_lang,original_content,original_question,original_answer,content,question,answer,language,translated,q_id,title,url,eclektic_id
0,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,Marquis of Wu'an is a feudal title used in sev...,Who was conferred the title of Marquis of Wu'a...,Tian Fen,en,1,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,en_502
1,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,Le titre de Marquis de Wu'an a été utilisé dan...,Qui a été fait Marquis de Wu'an sous la dynast...,Tian Fen,fr,1,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,fr_502
2,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,ווֹאָן הוֹאוּ הוא תואר אצולה שהיה בשימוש במספר...,"מי קיבל את התואר ""המרקיז של ווֹאָן"" בתקופת שוש...",טְייֵן פֿוּן,he,1,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,he_502
3,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,zh,0,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,zh_502
4,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,"무안후는 여러 왕조에서 사용되었던 작위명으로, 다음 인물을 지칭할 수 있습니다.\n...",서한은 누구를 무안후로 봉했습니까?,전분,ko,1,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,ko_502


In [39]:
print(df["eclektic_id"].nunique())

1512


In [4]:
filtered_tokens_array = []

# HEBREW KEYWORDS


In [5]:
he_df = df[df["language"] == "he"]
he_df.shape

(216, 13)

In [6]:
from transformers import BertModel, BertTokenizerFast

alephbert_tokenizer = BertTokenizerFast.from_pretrained('onlplab/alephbert-base')
alephbert = BertModel.from_pretrained('onlplab/alephbert-base')

# if not finetuning - disable dropout
alephbert.eval()


/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of BertModel were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(52000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [7]:
he_keep_pos = {
  "NOUN",   # common nouns
  "PROPN",  # proper nouns / names (matches zh: nr/ns/nt and NER-like usefulness)
  "VERB",   # verbs
  "ADJ",    # adjectives
  "ADV",    # adverbs
  "NUM"    # numbers (years, counts)
}

from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('dicta-il/dictabert-morph')
model = AutoModel.from_pretrained('dicta-il/dictabert-morph', trust_remote_code=True)

model.eval()

# iterate each row in df, tokenize the 'question' column, and keep only tokens with POS in he_keep_pos



for index, row in he_df.iterrows():
    sentence = row['question']
    result = model.predict([sentence], tokenizer)
    # keep only tokens with POS in he_keep_pos
    filtered_tokens = [token for token in result[0].get("tokens", []) if token['pos'] in he_keep_pos]
    eclektic_id = row[f"eclektic_id"]
    # save it accumulated in a dictionary with key as eclektic_id and value as filtered_tokens list
    filtered_tokens_array.append({"eclektic_id": eclektic_id, "question": sentence, "tokens": filtered_tokens})




In [ ]:
# save filtered_tokens_array to json file
with open('/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/processed/tokenization/filtered_tokens_array_he.json', 'w') as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)

# get the number of entries in filtered_tokens_array
print(f"Number of entries in filtered_tokens_array: {len(filtered_tokens_array)}")

Number of entries in filtered_tokens_array: 216


# JA POS

In [60]:
ja_df = df[df["language"] == "ja"]
ja_df.shape

(216, 13)

In [61]:
filtered_tokens_array = []

In [59]:
sentence = "西漢は誰を武安侯に封じましたか？"
m = tokenizer_obj.tokenize(sentence, SplitMode.C)
# filter to non-functional POS
for m in m:
    print(m.surface(), m.part_of_speech())

西 ('名詞', '普通名詞', '一般', '*', '*', '*')
漢 ('接尾辞', '名詞的', '一般', '*', '*', '*')
は ('助詞', '係助詞', '*', '*', '*', '*')
誰 ('代名詞', '*', '*', '*', '*', '*')
を ('助詞', '格助詞', '*', '*', '*', '*')
武安 ('名詞', '固有名詞', '人名', '姓', '*', '*')
侯 ('接尾辞', '名詞的', '一般', '*', '*', '*')
に ('助詞', '格助詞', '*', '*', '*', '*')
封じ ('動詞', '一般', '*', '*', 'サ行変格', '連用形-一般')
まし ('助動詞', '*', '*', '*', '助動詞-マス', '連用形-一般')
た ('助動詞', '*', '*', '*', '助動詞-タ', '終止形-一般')
か ('助詞', '終助詞', '*', '*', '*', '*')
？ ('補助記号', '句点', '*', '*', '*', '*')


In [62]:
ja_non_functional_pos = [
"名詞",
"動詞",
"形容詞",
"副詞",
]
tokenizer_obj = dictionary.Dictionary().create()
for index, row in ja_df.iterrows():
    sentence = row['question']
    m = tokenizer_obj.tokenize(sentence, SplitMode.C)
    # filter to non-functional POS
    tokens = []
    for m in m:
        if m.part_of_speech()[0] in ja_non_functional_pos:
            tokens.append({"token": m.surface(), "pos": m.part_of_speech()})
    filtered_tokens_array.append({"eclektic_id": eclektic_id, "question": sentence, "tokens": tokens})



In [63]:
len(filtered_tokens_array)

216

In [25]:
# get the number of unique eclektic_ids we have in the filtered_tokens_array
num_unique_eclektic_ids = len(set(item["eclektic_id"] for item in filtered_tokens_array))
print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")

Number of unique eclektic_ids with filtered tokens: 216


In [64]:

with open('/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/processed/tokenization/filtered_tokens_array_ja.json', 'w') as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)

# KO POS

In [65]:
ko_df = df[df["language"] == "ko"]
ko_df.shape

(216, 13)

In [66]:
filtered_tokens_array = []

In [67]:
preserve_pos = [
"Noun",
"Verb",
"Adjective",
"Adverb",
"Number",
"Foreign",
"Alpha"
]

okt = Okt()

for index, row in ko_df.iterrows():
    sentence = row['question']
    # tokens = okt.morphs(sentence)
    pos_tags = okt.pos(sentence)
    tokens = []
    for token, pos in pos_tags:
        if pos in preserve_pos:
            tokens.append({"token": token, "pos": pos})
    filtered_tokens_array.append({"eclektic_id": row[f"eclektic_id"], "question": sentence, "tokens": tokens})



In [68]:
len(filtered_tokens_array)  

216

In [70]:
# get the number of unique eclektic_ids we have in the filtered_tokens_array
num_unique_eclektic_ids = len(set(item["eclektic_id"] for item in filtered_tokens_array))
print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")

Number of unique eclektic_ids with filtered tokens: 216


In [69]:

with open('/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/processed/tokenization/filtered_tokens_array_ko.json', 'w') as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)   

# ZH POS

In [71]:
zh_df = df[df["language"] == "zh"]
zh_df.shape

(216, 13)

In [72]:
zh_non_functional_pos = [
    "n","f","s","t",
    "nr","ns","nt","nw","nz",
    "v","vd","vn",
    "a","ad","an",
    "d",
    "m","q","eng",
 
    "PER","LOC","ORG","TIME"
]

In [19]:
# sentence = "光绪二十九年对应西元哪一年？"
# words = pseg.cut(sentence, use_paddle=True)  # if you add vi mapping
# print("Using jieba.posseg with Paddle:")
# print(words)
# for tok in words:
#     # show word,flag only if it's in our non-functional POS list
#     if tok.flag in zh_non_functional_pos:   
#         print(f"{tok.word} ({tok.flag})")

In [73]:
filtered_tokens_array = []

In [74]:
zh_non_functional_pos = [
    "n","f","s","t",
    "nr","ns","nt","nw","nz",
    "v","vd","vn",
    "a","ad","an",
    "d",
    "m","q","eng",
    "PER","LOC","ORG","TIME"
]

for index, row in zh_df.iterrows():
    sentence = row['question']
    words = pseg.cut(sentence, use_paddle=True)  # if you add vi mapping
    tokens = []
    for tok in words:
        # show word,flag only if it's in our non-functional POS list
        if tok.flag in zh_non_functional_pos:   
            eclektic_id = row[f"eclektic_id"]
            tokens.append({"token": tok.word, "pos": tok.flag})
    filtered_tokens_array.append({"eclektic_id": eclektic_id, "question": sentence, "tokens": tokens})


In [75]:
len(filtered_tokens_array)

216

In [31]:
# get the number of unique eclektic_ids we have in the filtered_tokens_array
num_unique_eclektic_ids = len(set(item["eclektic_id"] for item in filtered_tokens_array))
print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")

Number of unique eclektic_ids with filtered tokens: 216


In [76]:

with open('/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/processed/tokenization/filtered_tokens_array_zh.json', 'w') as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)   

In [22]:

import json
with open("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/src/wimbd_cooc_features/pos_ner_keywords/pos_tokens.json", "w") as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)   

# for Eng, Fr, Es POS

In [77]:
import stanza

langs = ["en", "fr", "es"]

pipelines = {
    lang: stanza.Pipeline(lang, processors="tokenize,pos", use_gpu=False)
    for lang in langs
}

# text = "Paris est la capitale de la France."

# doc = pipelines["fr"](text)

# for sent in doc.sentences:
#     for word in sent.words:
#         print(word.text, word.upos)

2026-03-12 16:36:26 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-12 16:36:27 INFO: Downloaded file to /Users/anniewang/stanza_resources/resources.json
2026-03-12 16:36:27 WARNING: Language en package default expects mwt, which has been added
2026-03-12 16:36:27 INFO: Loading these models for language: en (English):
| Processor | Package         |
-------------------------------
| tokenize  | combined        |
| mwt       | combined        |
| pos       | combined_charlm |

2026-03-12 16:36:27 INFO: Using device: cpu
2026-03-12 16:36:27 INFO: Loading: tokenize
2026-03-12 16:36:27 INFO: Loading: mwt
2026-03-12 16:36:27 INFO: Loading: pos
2026-03-12 16:36:28 INFO: Done loading processors!
2026-03-12 16:36:28 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with downl

In [78]:
CONTENT_UPOS = {
"NOUN",
"PROPN",
"VERB",
"ADJ",
"ADV",
"NUM"
}

In [79]:
en_df = df[df["language"] == "en"]
es_df = df[df["language"] == "es"]
fr_df = df[df["language"] == "fr"]



In [80]:

print(en_df.shape)
print(es_df.shape)
print(fr_df.shape)

(216, 13)
(216, 13)
(216, 13)


In [ ]:
# # read json file
# with open("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/src/wimbd_cooc_features/pos_ner_keywords/pos_tokens.json", "r") as f:
#     filtered_tokens_array = json.load(f)
    
# # get the number of unique eclektic_ids we have in the filtered_tokens_array
# num_unique_eclektic_ids = len(set(item["eclektic_id"] for item in filtered_tokens_array))
# print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")


Number of unique eclektic_ids with filtered tokens: 864


In [81]:
filtered_tokens_array = []

In [82]:
for index, row in en_df.iterrows():
    sentence = row['question']
    doc = pipelines["en"](sentence)
    tokens = []
    for sent in doc.sentences:
        for word in sent.words:
            if word.upos in CONTENT_UPOS:
                eclektic_id = row[f"eclektic_id"]
                tokens.append({"token": word.text, "pos": word.upos})
    filtered_tokens_array.append({"eclektic_id": eclektic_id, "question": sentence, "tokens": tokens})

In [83]:
len(filtered_tokens_array)

216

In [84]:
# get the number of unique eclektic_ids we have in the filtered_tokens_array
num_unique_eclektic_ids = len(set(item["eclektic_id"] for item in filtered_tokens_array))
print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")

Number of unique eclektic_ids with filtered tokens: 216


In [85]:

with open('/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/processed/tokenization/filtered_tokens_array_en.json', 'w') as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)

In [86]:
filtered_tokens_array = []

In [87]:

for index, row in fr_df.iterrows():
    sentence = row['question']
    doc = pipelines["fr"](sentence)
    tokens = []
    for sent in doc.sentences:
        for word in sent.words:
            if word.upos in CONTENT_UPOS:
                eclektic_id = row[f"eclektic_id"]
                tokens.append({"token": word.text, "pos": word.upos})
    filtered_tokens_array.append({"eclektic_id": eclektic_id, "question": sentence, "tokens": tokens})

In [88]:
len(filtered_tokens_array)

216

In [89]:
# get the number of unique eclektic_ids we have in the filtered_tokens_array
num_unique_eclektic_ids = len(set(item["eclektic_id"] for item in filtered_tokens_array))
print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")

Number of unique eclektic_ids with filtered tokens: 216


In [90]:

with open('/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/processed/tokenization/filtered_tokens_array_fr.json', 'w') as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)

In [91]:
filtered_tokens_array = []

In [92]:

for index, row in es_df.iterrows():
    sentence = row['question']
    doc = pipelines["es"](sentence)
    tokens = []
    for sent in doc.sentences:
        for word in sent.words:
            if word.upos in CONTENT_UPOS:
                eclektic_id = row[f"eclektic_id"]
                tokens.append({"token": word.text, "pos": word.upos})
    filtered_tokens_array.append({"eclektic_id": eclektic_id, "question": sentence, "tokens": tokens})
            

In [93]:
len(filtered_tokens_array)

216

In [94]:
# get the number of unique eclektic_ids we have in the filtered_tokens_array
num_unique_eclektic_ids = len(set(item["eclektic_id"] for item in filtered_tokens_array))
print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")

Number of unique eclektic_ids with filtered tokens: 216


In [95]:

with open('/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/processed/tokenization/filtered_tokens_array_es.json', 'w') as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)

In [ ]:
import json
with open("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/src/wimbd_cooc_features/pos_ner_keywords/pos_tokens.json", "w") as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)   

In [ ]:
# load json and check the format
import json
with open("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/src/wimbd_cooc_features/pos_ner_keywords/pos_tokens.json", "r") as f:
    data = json.load(f)
    print(data[:2])

[{'eclektic_id': 'he_502', 'question': 'מי קיבל את התואר "המרקיז של ווֹאָן" בתקופת שושלת האן המערבית?', 'tokens': [{'token': 'קיבל', 'pos': 'VERB', 'feats': {'Gender': 'Masc', 'Number': 'Sing', 'Person': '3', 'Tense': 'Past'}, 'prefixes': [], 'suffix': False}, {'token': 'התואר', 'pos': 'NOUN', 'feats': {'Gender': 'Masc', 'Number': 'Sing'}, 'prefixes': ['DET'], 'suffix': False}, {'token': 'המרקיז', 'pos': 'NOUN', 'feats': {'Gender': 'Masc', 'Number': 'Sing'}, 'prefixes': ['DET'], 'suffix': False}, {'token': 'וואן', 'pos': 'PROPN', 'feats': {}, 'prefixes': [], 'suffix': False}, {'token': 'בתקופת', 'pos': 'NOUN', 'feats': {'Gender': 'Fem', 'Number': 'Sing'}, 'prefixes': ['ADP'], 'suffix': False}, {'token': 'שושלת', 'pos': 'NOUN', 'feats': {'Gender': 'Fem', 'Number': 'Sing'}, 'prefixes': [], 'suffix': False}, {'token': 'האן', 'pos': 'PROPN', 'feats': {}, 'prefixes': [], 'suffix': False}, {'token': 'המערבית', 'pos': 'ADJ', 'feats': {'Gender': 'Fem', 'Number': 'Sing'}, 'prefixes': ['DET'], '

In [ ]:
#count how many eclektic_ids we have in the data
print(len(data))

11549


In [ ]:
from tqdm import tqdm
SYSTEM_PROMPT = """
You are performing Named Entity Recognition (NER).

Allowed labels:
PERSON, ORG, LOCATION, DATE, TIME, MONEY, PERCENT, PRODUCT,
EVENT, WORK_OF_ART, LAW, LANGUAGE, NORP, FAC, ORDINAL,
CARDINAL, QUANTITY

Rules:
- Return JSON only
- Do not include markdown
- Use exact spans from the sentence
- Preserve multi-word entities
- Do not invent entities
- If none exist return {"entities": []}

Output format:
{
  "entities": [
    {"entity": "text", "ner": "label"}
  ]
}
"""

def extract_ner(sentence):

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Extract entities from this sentence:\n{sentence}",
        config=types.GenerateContentConfig(
            temperature=0,
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)



with tqdm(total=len(filtered_tokens_array), desc="Processing") as pbar:
    for q in filtered_tokens_array:
        sentence = q["question"]
        ner_result = extract_ner(sentence)
        q["ner"] = ner_result["entities"]
        pbar.update(1)

Processing:   1%|▏         | 173/11549 [06:16<5:28:31,  1.73s/it] 

In [ ]:
filtered_tokens_array[:5]

In [96]:
#load all json file from tokenization folder
import json
import os
tokenization_folder = "/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/processed/tokenization/"
all_token_files = [f for f in os.listdir(tokenization_folder) if f.endswith(".json")]
all_data = []
for file in all_token_files:
    with open(os.path.join(tokenization_folder, file), "r") as f:
        data = json.load(f)
        all_data.extend(data)

# append the lists together and save to a new json file
with open("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/src/wimbd_cooc_features/pos_ner_keywords/pos_tokens.json", "w") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=4)   

In [ ]:
import json
with open("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/src/wimbd_cooc_features/pos_ner_keywords/pos_tokens.json", "w") as f:
    json.dump(filtered_tokens_array, f, ensure_ascii=False, indent=4)   

In [97]:
#load json and check the format
import json
with open("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/src/wimbd_cooc_features/pos_ner_keywords/pos_tokens.json", "r") as f:
    data = json.load(f)
    print(len(data))

1512
